In [1]:
import os
from getpass import getpass

GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

llm_config = {
    "config_list": [
        {
            "model": "gemini-3.5-flash-lite",
            "api_type": "google",
            "api_key": GEMINI_API_KEY
        }
    ]
}
print("GEMINI configuration created successfully!")

Enter your Gemini API key:  ········


GEMINI configuration created successfully!


The Task

In [2]:
task = '''
        Write a concise but engaging blogpost about
      Abdul Kalam . Make sure the blogpost is
       within 100 words.
       '''

Create a writer agent

In [3]:
import autogen

writer = autogen.AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise " 
        "blogpost (with title) on given topics. You must polish your "
        "writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config=llm_config,
)

C:\Anaconda3\envs\agenticAI\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Anaconda3\envs\agenticAI\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.18) which Google will stop supporting in new releases of google.cloud.resourcemanager_v3 once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.cloud.resourcemanager_v3 past that date.
  warnings.warn(message, FutureWarning)


In [4]:
reply = writer.generate_reply(messages=[{"content": task,"role": "user"}])
print(reply)

C:\Anaconda3\envs\agenticAI\Lib\site-packages\autogen\oai\gemini.py:809: UserWarning: Cost calculation is not implemented for model gemini-3.5-flash-lite. Cost will be calculated zero.
  warnings.warn(


{'content': '**The People’s President: The Inspiring Legacy of A.P.J. Abdul Kalam**\n\nBorn into a humble family in Rameswaram, Dr. A.P.J. Abdul Kalam rose to become India’s celebrated "Missile Man" and the beloved 11th President of India. \n\nMore than a visionary scientist who shaped the nation\'s space and defense programs, he was a lifelong teacher who saw youth as India\'s greatest asset. He famously urged students to "dream, dream, dream, because dreams transform into thoughts and thoughts result in action." \n\nDr. Kalam left behind a legacy not of power, but of boundless curiosity, humility, and inspiration. His spark continues to light millions of minds today.', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


Adding reflection:
Create a critic agent to reflect on the work of the writer agent.

In [5]:
critic = autogen.AssistantAgent(
    name="Critic",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
    system_message="You are a critic. You review the work of "
                "the writer and provide constructive "
                "feedback to help improve the quality of the content.",
)

In [6]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
      Abdul Kalam . Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

# The People’s President: Lessons from A.P.J. Abdul Kalam

From launching India’s first space rockets to leading the nation as the beloved "People’s President," A.P.J. Abdul Kalam’s journey is nothing short of legendary. Born into a modest family, he rose through sheer brilliance and humility to become the architect of India's missile technology. 

Yet, beyond his scientific accolades, Kalam’s true legacy lies in his passion for youth. He believed that dreams aren't what you see in sleep, but what keep you awake. Today, his timeless wisdom continues to inspire millions to look up at the stars while keeping their feet firmly on the ground.

--------------------------------------------------------------------------------
Critic (to Writ

Nested chat

In [7]:
SEO_reviewer = autogen.AssistantAgent(
    name="SEO_Reviewer",
    llm_config=llm_config,
    system_message="You are an SEO reviewer, known for "
        "your ability to optimize content for search engines, "
        "ensuring that it ranks well and attracts organic traffic. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [8]:
legal_reviewer = autogen.AssistantAgent(
    name="Legal_Reviewer",
    llm_config=llm_config,
    system_message="You are a legal reviewer, known for "
        "your ability to ensure that content is legally compliant "
        "and free from any potential legal issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [9]:
ethics_reviewer = autogen.AssistantAgent(
    name="Ethics_Reviewer",
    llm_config=llm_config,
    system_message="You are an ethics reviewer, known for "
        "your ability to ensure that content is ethically sound "
        "and free from any potential ethical issues. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role. ",
)

In [10]:
meta_reviewer = autogen.AssistantAgent(
    name="Meta_Reviewer",
    llm_config=llm_config,
    system_message="You are a meta reviewer, you aggragate and review "
    "the work of other reviewers and give a final suggestion on the content.",
)

Orchestrate the nested chats to solve the task

In [11]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

review_chats = [
    {
     "recipient": SEO_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",},
     "max_turns": 1},
    {
    "recipient": legal_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}.",},
     "max_turns": 1},
    {"recipient": ethics_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'reviewer': '', 'review': ''}",},
     "max_turns": 1},
     {"recipient": meta_reviewer, 
      "message": "Aggregrate feedback from all reviewers and give final suggestions on the writing.", 
     "max_turns": 1},
]

In [12]:
critic.register_nested_chats(
    review_chats,
    trigger=writer, #final agent
)

In [13]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
      Abdul Kalam . Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

# The People’s President: Lessons from A.P.J. Abdul Kalam

From launching India’s first space rockets to leading the nation as the beloved "People’s President," A.P.J. Abdul Kalam’s journey is nothing short of legendary. Born into a modest family, he rose through sheer brilliance and humility to become the architect of India's missile technology. 

Yet, beyond his scientific accolades, Kalam’s true legacy lies in his passion for youth. He believed that dreams aren't what you see in sleep, but what keep you awake. Today, his timeless wisdom continues to inspire millions to look up at the stars while keeping their feet firmly on the ground.

--------------------------------------------------------------------------------

**************

In [14]:
print(res.summary)

# The People’s President: A.P.J. Abdul Kalam Biography and Legacy

### Meta Data
* **Title Tag:** A.P.J. Abdul Kalam Biography: The People's President
* **Meta Description:** Explore the inspiring life of A.P.J. Abdul Kalam, India's Missile Man and the beloved People's President, tracing his journey from humble roots to national icon.

---

## Early Life and Humble Beginnings

Born on October 15, 1931, in Rameswaram, Avul Pakir Jainulabdeen Abdul Kalam grew up in a modest Tamil family. To support his family during childhood, he distributed newspapers. Despite financial hardships, his insatiable curiosity and dedication to education propelled him forward. He went on to study physics at St. Joseph's College, Tiruchirappalli, and later earned a degree in aerospace engineering from the Madras Institute of Technology, laying the groundwork for a historic scientific career.

## The Missile Man: Scientific Career and ISRO/DRDO Contributions

Dr. Kalam's technical brilliance reshaped India’s d